# 🚀 L3M Studio × Google Colab Pro — 遠端 ComfyUI 高畫質影片／畫圖算力節點

本筆記本專為 **L3M Studio** 設計，讓您運用 **Google Colab Pro（L4 24GB / A100 40GB–80GB）** 作為遠端 ComfyUI 算力後台，流暢生成 **LTX-2.5 22B Distilled INT8**、**MiniMax-H3 FL2VA INT8（含 8步 Turbo）** 有聲影片與 **Z-Image-Turbo BF16** 高畫質圖片。

### 使用步驟
1. 點擊上方選單 **執行階段 (Runtime) → 變更執行階段類型**，選擇 **L4 GPU**（24GB VRAM，最省運算單元）或 **A100 GPU**（40GB/80GB，極速渲染）。
2. 依序執行下方 **步驟 1 → 步驟 4**。
3. 複製步驟 4 輸出的 **`https://xxxx.trycloudflare.com`** 網址（以及若您有設定的 `AUTH_TOKEN`）。
4. 回到電腦上的 **L3M Studio → 影片／畫圖頁**，將 ComfyUI 節點切到 **「雲端 Colab」** 並貼上該網址即可直接遠端生片，成品會自動下載回您本機 `D:\projects\locmodel-studio\data\outputs`！

> 💡 **優勢**：切換為「雲端 Colab」時，本機筆電顯卡完全不佔用，您可一邊在本機跟 Qwen3.6 聊天，一邊讓 Colab A100/L4 在雲端高速渲染影片。

In [ ]:
#@title 步驟 1：檢查 Colab GPU 與硬碟空間
import subprocess, shutil
print("=== GPU 資訊 ===")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=False)
total, used, free = shutil.disk_usage("/content")
print(f"\n=== /content 本機高速 SSD 空間 ===")
print(f"總計: {total/1e9:.1f} GB | 已用: {used/1e9:.1f} GB | 可用: {free/1e9:.1f} GB")

In [ ]:
#@title 步驟 2：安裝最新版 ComfyUI、官方工作流範本與 Cloudflare 通道
USE_GOOGLE_DRIVE_CACHE = False #@param {type:"boolean"} — 勾選可將模型快取到 Google Drive（注意：Colab 本機 SSD 下載極快，一般建議取消勾選以省 Drive 空間）
DRIVE_MODELS_DIR = "/content/drive/MyDrive/L3M_ComfyUI_Models" #@param {type:"string"}

import os, subprocess

# 安裝 aria2（多線程高速下載權重）與 cloudflared
subprocess.run("apt-get -y -qq update && apt-get -y -qq install aria2 ffmpeg", shell=True, check=True)
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
        shell=True, check=True
    )

COMFY_DIR = "/content/ComfyUI"
if not os.path.exists(COMFY_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", COMFY_DIR], check=True)
else:
    subprocess.run(["git", "-C", COMFY_DIR, "pull", "--ff-only"], check=False)

subprocess.run(["pip", "install", "-q", "-r", f"{COMFY_DIR}/requirements.txt", "comfyui-workflow-templates", "aiohttp"], check=True)

# 選用：連結 Google Drive 模型快取目錄
MODELS_ROOT = f"{COMFY_DIR}/models"
if USE_GOOGLE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
    for sub in ["diffusion_models", "text_encoders", "vae", "latent_upscale_models", "loras", "checkpoints"]:
        drv_sub = os.path.join(DRIVE_MODELS_DIR, sub)
        loc_sub = os.path.join(MODELS_ROOT, sub)
        os.makedirs(drv_sub, exist_ok=True)
        if os.path.exists(loc_sub) and not os.path.islink(loc_sub):
            subprocess.run(["rm", "-rf", loc_sub], check=True)
        if not os.path.islink(loc_sub):
            os.symlink(drv_sub, loc_sub)
    print(f"✅ 已掛載 Google Drive 模型目錄：{DRIVE_MODELS_DIR}")

print("✅ ComfyUI 與 cloudflared 安裝完成！")

In [ ]:
#@title 步驟 3：高速下載 L3M Studio 對應的高精度模型權重
DOWNLOAD_LTX25 = True #@param {type:"boolean"} — LTX-2.5 22B Distilled INT8（雙階超分有聲影片·推薦）
DOWNLOAD_MINIMAX_H3 = False #@param {type:"boolean"} — MiniMax-H3 FL2VA INT8 + 8步 Turbo LoRA（首尾幀有聲影片）
DOWNLOAD_Z_IMAGE_TURBO = False #@param {type:"boolean"} — Z-Image-Turbo BF16（8步高速高畫質繪圖）
HF_TOKEN = "" #@param {type:"string"} — 選填，若有需要授權的模型可填入 hf_...

import os, subprocess

MODELS_ROOT = "/content/ComfyUI/models"

def dl(url: str, subdir: str):
    dest_dir = os.path.join(MODELS_ROOT, subdir)
    os.makedirs(dest_dir, exist_ok=True)
    fname = url.split("/")[-1]
    dest = os.path.join(dest_dir, fname)
    if os.path.exists(dest) and os.path.getsize(dest) > 10_000_000:
        print(f"⏭️ 已存在：{subdir}/{fname}")
        return
    print(f"⬇️ 正在下載：{subdir}/{fname} ...")
    cmd = [
        "aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M",
        "--console-log-level=warn", "--summary-interval=10",
        "-d", dest_dir, "-o", fname,
    ]
    if HF_TOKEN.strip():
        cmd.extend(["--header", f"Authorization: Bearer {HF_TOKEN.strip()}"])
    cmd.append(url)
    subprocess.run(cmd, check=True)
    print(f"✅ 完成：{subdir}/{fname}")

if DOWNLOAD_LTX25:
    print("\n=== 下載 LTX-2.5 22B Distilled INT8 影片套件 ===")
    dl("https://huggingface.co/Lightricks/LTX-2.5/resolve/main/diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors", "diffusion_models")
    dl("https://huggingface.co/Lightricks/LTX-2.5/resolve/main/text_encoders/gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors", "text_encoders")
    dl("https://huggingface.co/Lightricks/LTX-2.5/resolve/main/vae/ltx-2.5-video-vae-bf16.safetensors", "vae")
    dl("https://huggingface.co/Lightricks/LTX-2.5/resolve/main/vae/ltx-2.5-audio-vae-bf16.safetensors", "vae")
    dl("https://huggingface.co/Lightricks/LTX-2.5/resolve/main/latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors", "latent_upscale_models")

if DOWNLOAD_MINIMAX_H3:
    print("\n=== 下載 MiniMax-H3 FL2VA INT8 影片套件 ===")
    dl("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "diffusion_models")
    dl("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "text_encoders")
    dl("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_int8_convrot.safetensors", "vae")
    dl("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors", "vae")
    dl("https://huggingface.co/lightx2v/Minimax-h3-Turbo/resolve/main/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", "loras")

if DOWNLOAD_Z_IMAGE_TURBO:
    print("\n=== 下載 Z-Image-Turbo BF16 繪圖套件 ===")
    dl("https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/diffusion_models/z_image_turbo_bf16.safetensors", "diffusion_models")
    dl("https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors", "text_encoders")
    dl("https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors", "vae")

print("\n🎉 所選模型皆已就緒！請繼續執行步驟 4 啟動服務。")

In [ ]:
#@title 步驟 4：啟動 ComfyUI 並產生專屬 HTTPS 連線網址（供 L3M Studio 貼上）
AUTH_TOKEN = "" #@param {type:"string"} — 選填安全密鑰（若填寫，需在 L3M Studio 的「遠端 Colab 認證 Token」填寫相同字串，避免他人誤用您的 GPU）
EXTRA_COMFY_ARGS = "--fast" #@param {type:"string"}

import os, re, subprocess, threading, time, urllib.request

COMFY_PORT = 8188
TUNNEL_TARGET_PORT = COMFY_PORT

# 若設定了 AUTH_TOKEN，在 8189 埠啟動輕量認證反向代理保護 8188 埠的 ComfyUI
if AUTH_TOKEN.strip():
    TUNNEL_TARGET_PORT = 8189
    proxy_code = f'''
import asyncio
from aiohttp import web, ClientSession, ClientTimeout
TOKEN = {AUTH_TOKEN.strip()!r}
TARGET = "http://127.0.0.1:{COMFY_PORT}"
async def handle(req):
    auth = req.headers.get("Authorization", "")
    xkey = req.headers.get("X-API-Key", "")
    if auth != f"Bearer {{TOKEN}}" and xkey != TOKEN:
        return web.Response(status=401, text="Unauthorized: invalid L3M Studio AUTH_TOKEN")
    url = TARGET + req.rel_url.raw_path_qs
    body = await req.read()
    hdrs = {{k: v for k, v in req.headers.items() if k.lower() not in ("host", "authorization", "x-api-key", "content-length")}}
    async with ClientSession(timeout=ClientTimeout(total=600)) as sess:
        async with sess.request(req.method, url, headers=hdrs, data=body) as resp:
            data = await resp.read()
            out_hdrs = {{k: v for k, v in resp.headers.items() if k.lower() not in ("content-encoding", "transfer-encoding", "content-length")}}
            return web.Response(status=resp.status, body=data, headers=out_hdrs)
app = web.Application(client_max_size=1024**3)
app.router.add_route("*", "/{{tail:.*}}", handle)
web.run_app(app, host="127.0.0.1", port={TUNNEL_TARGET_PORT}, print=None)
'''
    with open("/tmp/l3m_auth_proxy.py", "w") as f:
        f.write(proxy_code)
    subprocess.Popen(["python", "/tmp/l3m_auth_proxy.py"])
    print(f"🔐 已啟用認證保護代理（127.0.0.1:{TUNNEL_TARGET_PORT} -> :{COMFY_PORT}）")

# 1. 啟動 ComfyUI
cmd = ["python", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT)] + EXTRA_COMFY_ARGS.split()
comfy_proc = subprocess.Popen(cmd, cwd="/content/ComfyUI", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

def _stream_comfy():
    for line in comfy_proc.stdout:
        print(line, end="")

threading.Thread(target=_stream_comfy, daemon=True).start()

print("⏳ 正在等待 ComfyUI 啟動...")
for _ in range(90):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=2) as r:
            if r.status == 200:
                break
    except Exception:
        time.sleep(1.5)
else:
    raise RuntimeError("ComfyUI 啟動逾時，請查看上方日誌")

# 2. 啟動 Cloudflare Quick Tunnel
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{TUNNEL_TARGET_PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

tunnel_url = ""
for line in cf_proc.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        tunnel_url = m.group(0)
        print("\n" + "=" * 66)
        print("✅ L3M Studio 遠端 Colab ComfyUI 已連線就緒！")
        print(f"👉 遠端網址：{tunnel_url}")
        if AUTH_TOKEN.strip():
            print(f"🔐 認證 Token：{AUTH_TOKEN.strip()}")
        print("請將上方網址貼入 L3M Studio「影片／畫圖頁 → 雲端 Colab」或「設定」即可使用！")
        print("=" * 66 + "\n")
        break

# 保持執行階段運作並顯示 ComfyUI 渲染進度
try:
    comfy_proc.wait()
except KeyboardInterrupt:
    comfy_proc.terminate()
    cf_proc.terminate()